# Machine Learning Project: Fetal Health Classification

## A Beginner's Guide to Building ML Models for Disease Detection

---

**Project Overview:** This notebook demonstrates how to build a machine learning application that predicts fetal health outcomes from Cardiotocogram (CTG) data. We will follow a structured approach covering data exploration, model selection, implementation, and evaluation.

---

## Phase 1: Dataset Selection

### Dataset Summary

| Attribute | Details |
|-----------|--------|
| **Dataset Name** | Fetal Health Classification |
| **Source** | Kaggle (originally from UCI Machine Learning Repository) |
| **Problem Description** | Classification of fetal health to prevent child and maternal mortality |
| **Target Variable** | fetal_health (1 = Normal, 2 = Suspect, 3 = Pathological) |
| **Number of Rows** | 2,126 records |
| **Number of Columns** | 22 features |

### Why This Dataset?

This dataset addresses a **real-world healthcare problem**. According to the World Health Organization (WHO), reducing child and maternal mortality is a critical global health goal. Cardiotocogram (CTG) is a non-invasive monitoring technique used during pregnancy and labor to assess fetal well-being. By classifying fetal health status, we can help medical professionals take timely interventions.

### Dataset Features

The dataset contains features extracted from CTG exams including:
- **baseline value**: Fetal heart rate baseline (FHR)
- **accelerations**: Number of accelerations per second
- **fetal_movement**: Number of fetal movements per second
- **uterine_contractions**: Number of uterine contractions per second
- **light_decelerations**: Number of light decelerations per second
- **severe_decelerations**: Number of severe decelerations per second
- **prolongued_decelerations**: Number of prolonged decelerations per second
- **abnormal_short_term_variability**: Percentage of time with abnormal short term variability
- **mean_value_of_short_term_variability**: Mean value of short term variability
- And more histogram-based features...

## Phase 2: Problem Definition

### What the System Will Predict

The system will predict the **fetal health status** into one of three categories:
- **Class 1 (Normal)**: Healthy fetal condition
- **Class 2 (Suspect)**: Requires closer monitoring
- **Class 3 (Pathological)**: Requires immediate medical attention

### Type of Machine Learning Problem

This is a **Multi-class Classification** problem because:
1. We have a categorical target variable with 3 distinct classes
2. We want to predict which category each sample belongs to
3. There is no natural ordering between the classes

### Input Features and Target Variable

- **Input Features (X)**: 21 numerical features from CTG measurements
- **Target Variable (y)**: fetal_health (1, 2, or 3)

### Why This Problem is Important

1. **Critical Healthcare Impact**: Early detection of fetal distress can save lives
2. **Preventive Medicine**: Enables timely interventions before complications
3. **Scalable Solution**: ML models can assist doctors in making faster decisions
4. **Global Health Goal**: Aligns with UN Sustainable Development Goals for reducing child mortality

### How Machine Learning Can Solve It

Machine learning can analyze patterns in CTG data that might be subtle or complex for human interpretation. By training on historical data with known outcomes, ML models can:
- Learn the relationship between CTG features and fetal health
- Identify high-risk patterns automatically
- Provide rapid, consistent assessments
- Scale to support healthcare systems with limited specialists

## Phase 3: Data Exploration and Preparation

Let's start by loading and exploring the dataset.

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print("Libraries imported successfully!")

In [ ]:
# Load the dataset
# The dataset should be in the same directory or provide the full path
df = pd.read_csv('/home/z/my-project/upload/data.csv')

# Display basic information
print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)
print(f"\nDataset Shape: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"\nColumn Names:\n{list(df.columns)}")

In [ ]:
# Display first few rows
print("First 5 rows of the dataset:")
df.head()

In [ ]:
# Display dataset information
print("Dataset Information:")
df.info()

In [ ]:
# Statistical summary
print("Statistical Summary:")
df.describe()

### 3.1 Analyzing Missing Values

Missing values can significantly impact model performance. Let's check for any missing data.

In [ ]:
# Check for missing values
missing_values = df.isnull().sum()
missing_percentage = (missing_values / len(df)) * 100

missing_df = pd.DataFrame({
    'Column': df.columns,
    'Missing Values': missing_values.values,
    'Percentage (%)': missing_percentage.values
})

print("Missing Values Analysis:")
print("=" * 50)
print(f"Total missing values in dataset: {missing_values.sum()}")
print(f"\nColumns with missing values:")
missing_df[missing_df['Missing Values'] > 0]

# Visualize missing values
if missing_values.sum() > 0:
    plt.figure(figsize=(12, 6))
    plt.bar(missing_df['Column'], missing_df['Missing Values'])
    plt.xticks(rotation=90)
    plt.title('Missing Values by Column')
    plt.ylabel('Number of Missing Values')
    plt.tight_layout()
    plt.show()
else:
    print("\nNo missing values found in the dataset!")

### 3.2 Analyzing Data Types

Understanding data types helps us decide on preprocessing steps.

In [ ]:
# Data types analysis
print("Data Types Summary:")
print("=" * 50)
print(df.dtypes.value_counts())
print("\nDetailed Data Types:")
for col in df.columns:
    print(f"{col}: {df[col].dtype}")

### 3.3 Target Variable Analysis

Let's examine the distribution of our target variable - fetal_health.

In [ ]:
# Target variable distribution
target_counts = df['fetal_health'].value_counts()
target_labels = {1: 'Normal', 2: 'Suspect', 3: 'Pathological'}

print("Target Variable Distribution:")
print("=" * 50)
for value, count in target_counts.items():
    print(f"{target_labels[value]} (Class {value}): {count} samples ({count/len(df)*100:.2f}%)")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar plot
colors = ['#2ecc71', '#f39c12', '#e74c3c']
axes[0].bar([target_labels[i] for i in target_counts.index], target_counts.values, color=colors)
axes[0].set_title('Fetal Health Distribution (Bar Plot)', fontsize=12)
axes[0].set_xlabel('Fetal Health Status')
axes[0].set_ylabel('Number of Samples')
for i, v in enumerate(target_counts.values):
    axes[0].text(i, v + 10, str(v), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(target_counts.values, labels=[target_labels[i] for i in target_counts.index], 
            autopct='%1.1f%%', colors=colors, explode=(0.02, 0.02, 0.02), startangle=90)
axes[1].set_title('Fetal Health Distribution (Pie Chart)', fontsize=12)

plt.tight_layout()
plt.show()

print("\nNote: The dataset is imbalanced with more 'Normal' cases than 'Suspect' or 'Pathological'.")
print("This is common in medical datasets and needs to be considered during model evaluation.")

### 3.4 Correlation Analysis

Correlation helps us understand relationships between features and identify potential multicollinearity.

In [ ]:
# Correlation heatmap
plt.figure(figsize=(16, 12))
correlation_matrix = df.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, 
            fmt='.2f', linewidths=0.5, annot_kws={'size': 8})
plt.title('Correlation Heatmap of All Features', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation with target variable
target_correlation = correlation_matrix['fetal_health'].drop('fetal_health').sort_values(ascending=False)

plt.figure(figsize=(12, 8))
colors = ['#e74c3c' if x > 0 else '#3498db' for x in target_correlation.values]
bars = plt.barh(target_correlation.index, target_correlation.values, color=colors)
plt.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
plt.title('Feature Correlation with Fetal Health', fontsize=14)
plt.xlabel('Correlation Coefficient')
plt.ylabel('Features')

# Add correlation values on bars
for bar, val in zip(bars, target_correlation.values):
    plt.text(val + 0.01 if val > 0 else val - 0.03, bar.get_y() + bar.get_height()/2, 
             f'{val:.3f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

print("\nTop 5 Positive Correlations:")
print(target_correlation.head())
print("\nTop 5 Negative Correlations:")
print(target_correlation.tail())

### 3.5 Feature Distributions

Let's examine the distribution of key features to understand their spread and identify potential outliers.

In [ ]:
# Distribution plots for key features
key_features = ['baseline value', 'accelerations', 'fetal_movement', 'uterine_contractions', 
                'abnormal_short_term_variability', 'mean_value_of_short_term_variability',
                'histogram_mean', 'histogram_median']

fig, axes = plt.subplots(2, 4, figsize=(18, 10))
axes = axes.flatten()

for i, feature in enumerate(key_features):
    axes[i].hist(df[feature], bins=30, color='#3498db', edgecolor='black', alpha=0.7)
    axes[i].set_title(f'Distribution of {feature}', fontsize=11)
    axes[i].set_xlabel(feature)
    axes[i].set_ylabel('Frequency')

plt.suptitle('Distribution of Key Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Box plots for key features by fetal health
fig, axes = plt.subplots(2, 4, figsize=(18, 10))
axes = axes.flatten()

for i, feature in enumerate(key_features):
    df.boxplot(column=feature, by='fetal_health', ax=axes[i])
    axes[i].set_title(f'{feature} by Fetal Health', fontsize=10)
    axes[i].set_xlabel('Fetal Health (1=Normal, 2=Suspect, 3=Pathological)')
    axes[i].set_ylabel(feature)

plt.suptitle('Feature Distributions by Fetal Health Status', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 3.6 Outlier Detection

We'll use the Interquartile Range (IQR) method to detect outliers.

In [ ]:
# Outlier detection using IQR method
def detect_outliers_iqr(data, column):
    """Detect outliers using IQR method"""
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = data[(data[column] < lower_bound) | (data[column] > upper_bound)]
    return len(outliers), lower_bound, upper_bound

print("Outlier Analysis (IQR Method):")
print("=" * 70)

outlier_summary = []
for col in df.columns[:-1]:  # Exclude target variable
    count, lower, upper = detect_outliers_iqr(df, col)
    percentage = (count / len(df)) * 100
    outlier_summary.append({
        'Feature': col,
        'Outlier Count': count,
        'Percentage': f'{percentage:.2f}%'
    })

outlier_df = pd.DataFrame(outlier_summary)
print(outlier_df.to_string(index=False))

### 3.7 Data Preprocessing

Based on our analysis, we'll perform the following preprocessing steps:

1. **No missing values to handle** - The dataset is complete
2. **No categorical variables to encode** - All features are numerical
3. **Feature Scaling** - Apply StandardScaler for algorithms sensitive to feature scales
4. **Train-Test Split** - Split data for model training and evaluation

In [ ]:
# Separate features and target
X = df.drop('fetal_health', axis=1)
y = df['fetal_health']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

In [ ]:
# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Dataset Split:")
print("=" * 50)
print(f"Training set: {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Testing set: {X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.1f}%)")
print(f"\nClass distribution in training set:")
print(y_train.value_counts())
print(f"\nClass distribution in testing set:")
print(y_test.value_counts())

In [ ]:
# Feature scaling using StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrame for better visualization
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=X.columns)

print("Feature Scaling Applied (StandardScaler):")
print("=" * 50)
print("\nBefore Scaling (first 5 rows, first 3 columns):")
print(X_train.iloc[:5, :3])
print("\nAfter Scaling (first 5 rows, first 3 columns):")
print(X_train_scaled_df.iloc[:5, :3])

### Preprocessing Decisions Summary

| Decision | Reason |
|----------|--------|
| **No missing value handling needed** | Dataset has no missing values |
| **No categorical encoding needed** | All features are numerical |
| **Applied StandardScaler** | Ensures all features have mean=0 and std=1, important for SVM and Logistic Regression |
| **Used stratified split** | Maintains class proportions in train/test sets |
| **Kept all features** | All features are relevant medical measurements; no dimensionality reduction needed initially |

## Phase 4: Model Selection

For this multi-class classification problem, we will implement **three different algorithms**:

### 1. Logistic Regression

**Why Logistic Regression?**
- **Algorithm Strengths**: Simple, interpretable, fast training, provides probability outputs
- **Suitability for Dataset**: Works well with numerical features after scaling; good baseline model
- **Expected Advantages**: Quick to train, easy to interpret which features are important, performs well when classes are linearly separable

### 2. Random Forest Classifier

**Why Random Forest?**
- **Algorithm Strengths**: Handles non-linear relationships, robust to outliers, provides feature importance, less prone to overfitting than decision trees
- **Suitability for Dataset**: Can capture complex interactions between CTG features; doesn't require feature scaling
- **Expected Advantages**: High accuracy, handles imbalanced datasets better, interpretable feature importance

### 3. Support Vector Machine (SVM)

**Why SVM?**
- **Algorithm Strengths**: Effective in high-dimensional spaces, memory efficient, versatile with different kernel functions
- **Suitability for Dataset**: Works well with the 21 features in our dataset; can capture non-linear decision boundaries
- **Expected Advantages**: Good generalization, effective when features have clear margin of separation

## Phase 5: Model Implementation

Now let's implement and train all three models.

In [ ]:
# Initialize models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42, multi_class='multinomial'),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Support Vector Machine': SVC(kernel='rbf', random_state=42)
}

# Dictionary to store results
results = {}

print("Training Models...")
print("=" * 50)

# Train and evaluate each model
for name, model in models.items():
    print(f"\nTraining {name}...")
    
    # Use scaled data for Logistic Regression and SVM
    if name in ['Logistic Regression', 'Support Vector Machine']:
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted')
    recall = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')
    
    # Store results
    results[name] = {
        'model': model,
        'predictions': y_pred,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1_score': f1
    }
    
    print(f"{name} trained successfully!")
    print(f"Accuracy: {accuracy:.4f}")

In [ ]:
# Display detailed classification reports
for name, result in results.items():
    print(f"\n{'=' * 60}")
    print(f"Classification Report for {name}")
    print('=' * 60)
    print(classification_report(y_test, result['predictions'], 
                                target_names=['Normal', 'Suspect', 'Pathological']))

In [ ]:
# Confusion Matrices
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, (name, result) in enumerate(results.items()):
    cm = confusion_matrix(y_test, result['predictions'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i],
                xticklabels=['Normal', 'Suspect', 'Pathological'],
                yticklabels=['Normal', 'Suspect', 'Pathological'])
    axes[i].set_title(f'Confusion Matrix\n{name}', fontsize=12)
    axes[i].set_xlabel('Predicted')
    axes[i].set_ylabel('Actual')

plt.tight_layout()
plt.show()

## Phase 6: Model Evaluation and Comparison

Let's compare the performance of all three models.

In [ ]:
# Create comparison table
comparison_data = []
for name, result in results.items():
    comparison_data.append({
        'Model': name,
        'Accuracy': f"{result['accuracy']:.4f}",
        'Precision': f"{result['precision']:.4f}",
        'Recall': f"{result['recall']:.4f}",
        'F1-Score': f"{result['f1_score']:.4f}"
    })

comparison_df = pd.DataFrame(comparison_data)
print("\nModel Performance Comparison:")
print("=" * 70)
print(comparison_df.to_string(index=False))

In [ ]:
# Visualize model comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

metrics = ['accuracy', 'precision', 'recall', 'f1_score']
metric_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
colors = ['#3498db', '#2ecc71', '#e74c3c']

for i, (metric, metric_name) in enumerate(zip(metrics, metric_names)):
    ax = axes[i // 2, i % 2]
    values = [results[model][metric] for model in results.keys()]
    bars = ax.bar(list(results.keys()), values, color=colors)
    ax.set_title(f'{metric_name} Comparison', fontsize=12)
    ax.set_ylabel(metric_name)
    ax.set_ylim(0, 1.1)
    
    # Add value labels on bars
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
                f'{val:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    ax.tick_params(axis='x', rotation=15)

plt.suptitle('Model Performance Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Feature Importance from Random Forest
rf_model = results['Random Forest']['model']
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=True)

plt.figure(figsize=(12, 8))
bars = plt.barh(feature_importance['Feature'], feature_importance['Importance'], color='#2ecc71')
plt.title('Feature Importance (Random Forest)', fontsize=14)
plt.xlabel('Importance Score')
plt.ylabel('Feature')

# Add value labels
for bar, val in zip(bars, feature_importance['Importance'].values):
    plt.text(val + 0.005, bar.get_y() + bar.get_height()/2, 
             f'{val:.4f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

print("\nTop 5 Most Important Features:")
print(feature_importance.sort_values('Importance', ascending=False).head().to_string(index=False))

### Model Comparison Analysis

#### Best Performing Model

Based on our evaluation, we can identify the best performing model by considering multiple metrics:

- **Random Forest** typically performs well on this dataset because it can capture non-linear relationships between CTG features
- **Support Vector Machine** often shows competitive results due to its ability to find optimal decision boundaries
- **Logistic Regression** provides a good baseline with interpretable results

#### Strengths and Weaknesses

| Model | Strengths | Weaknesses |
|-------|-----------|------------|
| **Logistic Regression** | Fast, interpretable, provides probabilities | Assumes linear decision boundaries |
| **Random Forest** | High accuracy, handles non-linearity, feature importance | Can be slow on large datasets, less interpretable |
| **SVM** | Good generalization, handles high dimensions | Requires feature scaling, slow training |

#### Overfitting/Underfitting Analysis

To check for overfitting, we compare training and testing accuracy:

In [ ]:
# Check for overfitting by comparing train vs test accuracy
print("Overfitting Analysis (Train vs Test Accuracy):")
print("=" * 60)

for name, result in results.items():
    model = result['model']
    
    # Get training predictions
    if name in ['Logistic Regression', 'Support Vector Machine']:
        train_pred = model.predict(X_train_scaled)
    else:
        train_pred = model.predict(X_train)
    
    train_acc = accuracy_score(y_train, train_pred)
    test_acc = result['accuracy']
    
    print(f"\n{name}:")
    print(f"  Training Accuracy: {train_acc:.4f}")
    print(f"  Testing Accuracy:  {test_acc:.4f}")
    print(f"  Difference: {train_acc - test_acc:.4f}")
    
    if train_acc - test_acc > 0.1:
        print("  Status: ⚠️ Possible OVERFITTING (large gap between train and test)")
    elif train_acc < 0.7:
        print("  Status: ⚠️ Possible UNDERFITTING (low training accuracy)")
    else:
        print("  Status: ✅ Good generalization")

## Phase 7: Ethical and Professional Considerations

### Data Privacy

**Key Considerations:**

1. **Patient Confidentiality**: Medical data, including CTG recordings, contains sensitive health information. The original dataset should be anonymized to remove any personally identifiable information.

2. **Data Storage and Access**: In a real-world deployment, patient data must be stored securely with proper encryption and access controls. Only authorized medical personnel should have access.

3. **HIPAA Compliance**: In the United States, any healthcare application handling patient data must comply with the Health Insurance Portability and Accountability Act (HIPAA). Similar regulations exist globally (GDPR in Europe, PIPEDA in Canada).

### Bias in Datasets

**Potential Sources of Bias:**

1. **Class Imbalance**: Our dataset shows significant imbalance with ~78% Normal cases, ~14% Suspect, and ~8% Pathological. This can cause models to be biased toward predicting the majority class.

2. **Population Bias**: The dataset may not represent all populations equally. Factors like age, ethnicity, geographic location, and socioeconomic status can affect fetal health patterns.

3. **Equipment Bias**: CTG machines from different manufacturers may produce slightly different readings, potentially affecting model generalization.

**Mitigation Strategies:**
- Use techniques like SMOTE (Synthetic Minority Over-sampling) to address class imbalance
- Collect diverse training data from multiple sources
- Regularly validate model performance across different demographic groups

### Ethical Implications

1. **Medical Decision Support vs. Replacement**: ML models should assist, not replace, medical professionals. Doctors should make final decisions considering model predictions along with clinical judgment.

2. **Informed Consent**: Patients should be informed when ML tools are used in their care, including the limitations and potential risks.

3. **Accountability**: Clear guidelines must exist for who is responsible when ML predictions lead to incorrect medical decisions.

4. **Transparency**: Model decisions should be explainable, especially in healthcare where understanding "why" a prediction was made is crucial.

### Security Concerns

1. **Adversarial Attacks**: ML models in healthcare can be vulnerable to adversarial attacks that manipulate input data to cause incorrect predictions.

2. **Model Theft**: Proprietary models trained on expensive medical data could be stolen and misused.

3. **System Reliability**: The model must be robust and reliable, as errors can have serious health consequences.

### Legal Considerations

1. **Medical Device Regulations**: In many countries, ML-based diagnostic tools are classified as medical devices and must meet regulatory requirements (FDA approval in the US, CE marking in Europe).

2. **Liability**: Clear legal frameworks must define who is liable for misdiagnosis - the healthcare provider, the ML developer, or both.

3. **Data Rights**: Patients have rights over their medical data, including the right to know how it's used and the right to have it deleted.

## Conclusion and Recommendations

### Summary

In this project, we successfully developed a machine learning application for fetal health classification using CTG data. We:

1. **Selected** a real-world healthcare dataset addressing child mortality
2. **Defined** a multi-class classification problem with clear objectives
3. **Explored** the data through comprehensive EDA and visualizations
4. **Implemented** three different ML algorithms: Logistic Regression, Random Forest, and SVM
5. **Evaluated** model performance using multiple metrics
6. **Discussed** ethical considerations crucial for healthcare ML applications

### Key Findings

- Random Forest typically achieves the highest accuracy due to its ability to capture non-linear relationships
- Important features for prediction include abnormal_short_term_variability, histogram_mean, and histogram_width
- The dataset has class imbalance, which should be addressed for production deployment

### Recommendations for Improvement

1. **Address Class Imbalance**: Use SMOTE, class weights, or ensemble methods
2. **Hyperparameter Tuning**: Use GridSearchCV or RandomizedSearchCV for optimization
3. **Cross-Validation**: Implement k-fold cross-validation for more robust evaluation
4. **Deep Learning**: Explore neural networks for potentially better performance
5. **Feature Engineering**: Create new features or use PCA for dimensionality reduction

In [ ]:
# Final summary of best model
best_model_name = max(results.keys(), key=lambda x: results[x]['accuracy'])
best_result = results[best_model_name]

print("\n" + "=" * 70)
print("FINAL MODEL SUMMARY")
print("=" * 70)
print(f"\nBest Performing Model: {best_model_name}")
print(f"\nPerformance Metrics:")
print(f"  - Accuracy:  {best_result['accuracy']:.4f} ({best_result['accuracy']*100:.2f}%)")
print(f"  - Precision: {best_result['precision']:.4f}")
print(f"  - Recall:    {best_result['recall']:.4f}")
print(f"  - F1-Score:  {best_result['f1_score']:.4f}")
print("\n" + "=" * 70)
print("Thank you for reviewing this Machine Learning project!")
print("=" * 70)